# Older-year evaluation: 2016–2017

This is evaluation of frozen 2018–2020-trained models on earlier NASA S-NPP labels. It is **backward temporal and cross-sensor transfer**, not a forward operational backtest. No fitting, calibration, threshold changes or model selection occur.

Existing 2017 HLS inputs would contaminate 2016 evaluation. Separate **2015 HLS** observations replace image inputs while learned parameters remain fixed. Original S-NPP models trained on these years are excluded. See `docs/older-evaluation-results.md` and `docs/older-evaluation-statecharts.md`.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from firecast.benchmark_temporal import sha, evaluate
from firecast.older_evaluation import guard, evaluation_data, substitute_image_inputs

ROOT = Path.cwd()
if not (ROOT / 'outputs').exists():
    ROOT = ROOT.parent
config = json.loads((ROOT / 'configs/older-evaluation.json').read_text())
frozen = guard(ROOT, config)
report = json.loads((ROOT / 'outputs/older-evaluation/results.json').read_text())
assert report['model_weights_calibrators_thresholds_unchanged']
assert not report['production_approved']
print('Source:', report['source_product'], 'Training labels:', report['training_label_product'])
print('Candidates:', len(report['results']), 'Matched cells:', len(report['image_cells']))
print('Earliest fitting target:', report['manifest']['earliest_fitting_target'])
print('Source overlap audit:', report['manifest']['source_overlaps'])

Source: VNP14A1.002 Training labels: VJ114A1.002
Candidates: 14 Matched cells: 115
Earliest fitting target: 2018-01-05
Source overlap audit: {'soil_overlap_rows': 372, 'ndvi_duplicate_months': ['2017-09-01', '2017-10-01', '2017-11-01', '2017-12-01']}


## Cohorts and exclusions

Daily source panels retain all 124 cells, including southern Lebanon before the 2023 conflict cutoff. Cloud/unknown/coverage and historical FIRMS exclusions are preserved under strict and conservative policies. Missing past vegetation and missing image-domain cases do not become negatives.

To keep a 14-day gap before the earliest fitting target, 21–31 December 2017 are withheld. The remaining 2017 interval ends on 20 December. Modern model availability, retrospective processing and observed-weather proxies still prevent an operational-as-of claim.

In [2]:
rows=[]
for name, c in report['cohorts'].items():
    rows.append({'year_policy':name, **{'full_'+k:v for k,v in c['full_feature_qualified'].items()},
                 **{'matched_'+k:v for k,v in c['matched_images'].items()}})
pd.DataFrame(rows).set_index('year_policy')

,full_windows,full_positive_windows,full_image_scope_excluded_windows,full_image_scope_excluded_positive_windows,matched_windows,matched_positive_windows,matched_image_scope_excluded_windows,matched_image_scope_excluded_positive_windows
year_policy,,,,,,,,
2016-strict,39260,90,0,0,36587,78,2673,12
2016-conservative,39553,110,0,0,36792,91,2761,19
2017-strict,37730,36,0,0,35168,31,2562,5
2017-conservative,37939,40,0,0,35303,35,2636,5


In [3]:
pd.read_csv(ROOT / 'outputs/older-evaluation/results.csv').query("domain.str.endswith('strict-matched')",engine='python')[['model','domain','observations','positive_windows','pr_auc_average_precision','precision','recall','brier']]

,model,domain,observations,positive_windows,pr_auc_average_precision,precision,recall,brier
0,seasonal,2016-strict-matched,36587,78,0.003828,0.003943,0.679487,0.002125
2,seasonal,2017-strict-matched,35168,31,0.001317,0.001404,0.612903,0.000885
4,weather-logistic,2016-strict-matched,36587,78,0.005706,0.007083,0.141026,0.002124
6,weather-logistic,2017-strict-matched,35168,31,0.003290,0.002601,0.193548,0.000888
8,environment-logistic,2016-strict-matched,36587,78,0.010236,0.007985,0.500000,0.002118
10,environment-logistic,2017-strict-matched,35168,31,0.003618,0.002163,0.354839,0.000890
12,no-image-mlp,2016-strict-matched,36587,78,0.009807,0.007972,0.525641,0.002119
14,no-image-mlp,2017-strict-matched,35168,31,0.002797,0.002670,0.451613,0.000887
16,prithvi-v1-100m-frozen_encoder,2016-strict-matched,36587,78,0.011570,0.018692,0.025641,0.002119
18,prithvi-v1-100m-frozen_encoder,2017-strict-matched,35168,31,0.004707,0.009346,0.032258,0.000883


## Reproduce the previously selected external model without labels as inputs

Only image observations are replaced. The original head state, scalers, calibration and threshold remain unchanged; original model files are hash-checked afterward. Zero alert precision is not estimable, although metric JSON retains a zero convention with `precision_estimable=false`.

In [4]:
identifier = report['selected_external_unchanged']
record = frozen['artifacts'][identifier]
artifact_path = ROOT / 'outputs/pretrained-transfer' / record['file']
assert sha(artifact_path) == record['sha256']
artifact = joblib.load(artifact_path)
original_state_hash = joblib.hash(artifact['model'])
source_id = next(m['id'] for m in config['models'] if identifier.startswith(m['id']+'-'))
with np.load(ROOT / 'data/older-pretrained' / (source_id+'-tokens.npz')) as a:
    bank = {k:a[k] for k in a.files}
model = substitute_image_inputs(artifact['model'],bank)
with np.load(ROOT / 'outputs/older-evaluation-data/2016.npz') as a:
    data,audit=evaluation_data({k:a[k] for k in a.files},'strict',report['image_cells'])
probabilities=model.predict({k:v for k,v in data.items() if k!='y'})
measured=evaluate(data,probabilities,artifact['threshold'])
expected=report['results'][identifier]['2016-strict-matched']
assert np.isclose(measured['pr_auc_average_precision'],expected['pr_auc_average_precision'])
assert measured['confusion_matrix_tn_fp_fn_tp']==expected['confusion_matrix_tn_fp_fn_tp']
assert joblib.hash(artifact['model'])==original_state_hash
assert sha(artifact_path)==record['sha256']
guard(ROOT,config)
images=json.loads((ROOT/'data/older-pretrained/hls-patches/manifest.json').read_text())
assert all(r['observation_time']<'2016-01-01' for r in images['records'])
print('Reproduced AP:',measured['pr_auc_average_precision'])
print('TN/FP/FN/TP:',measured['confusion_matrix_tn_fp_fn_tp'])
print('Original fitted model unchanged:',True)

Reproduced AP: 0.010080025233327655
TN/FP/FN/TP: [36509, 0, 78, 0]
Original fitted model unchanged: True


```mermaid
stateDiagram-v2
    [*] --> VerifyFrozenModelsAndTrainingDates
    VerifyFrozenModelsAndTrainingDates --> SNPPLabels2016And2017
    SNPPLabels2016And2017 --> CloudAndLabelPolicies
    CloudAndLabelPolicies --> PastWeatherNDVIAnd2015HLS
    PastWeatherNDVIAnd2015HLS --> TemporalEmbargoAndImageMatching
    TemporalEmbargoAndImageMatching --> FrozenPredictions
    FrozenPredictions --> CompareWithOlderLabels
    CompareWithOlderLabels --> RetainOriginalSelections
    RetainOriginalSelections --> ProductionBlocked
```